In [1]:
import yaml
import torch
import modules
from xlstm.xlstm_large.model import xLSTMLargeConfig, xLSTMLarge
from tokenizers import Tokenizer
from torchviz import make_dot
from torchinfo import summary

# standardize relative filepaths
%cd ../

/mnt/ssd/Code/ArtI/SL


In [4]:
torch.set_default_device("cuda")

config_path = "./SL/configs/transformer_tinystories_small_v2.yaml"

# open config
with open(config_path) as ConfigFile:
    c = yaml.safe_load(ConfigFile)

# load tokenizer
tokenizer = Tokenizer.from_file(config["tokenizer_path"])
c["vocab_size"] = tokenizer.get_vocab_size()

# model
match(c["model_type"]):
    case "transformer":
        bot = modules.transformer(c)
    case "xlstm":
        # configure the model with TFLA Triton kernels
        xlstm_config = xLSTMLargeConfig(
            embedding_dim=c["d_model"],
            num_heads=c["num_heads"],
            num_blocks=c["num_layers"],
            vocab_size=c["vocab_size"],
            return_last_states=c["return_last_states"],
            chunkwise_kernel=c["chunkwise_kernel"], 
            sequence_kernel=c["sequence_kernel"],
            step_kernel=c["step_kernel"],
            mode="train",
        )
        # instantiate the model
        bot = xLSTMLarge(xlstm_config)
    case _:
        raise ValueError("unknown model type")

bot.eval()

print(summary(bot))

print(f"Parameters with weigth sharing: {sum(p.numel() for p in bot.parameters())}")
print(f"Trainable parameters with weigth sharing: {sum(p.numel() for p in bot.parameters() if p.requires_grad)}")

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Layer (type:depth-idx)                   Param #
transformer                              --
├─Embedding: 1-1                         524,288
├─Embedding: 1-2                         65,536
├─ModuleList: 1-3                        --
│    └─LayerNorm: 2-1                    256
│    └─LayerNorm: 2-2                    256
│    └─LayerNorm: 2-3                    256
│    └─LayerNorm: 2-4                    256
│    └─LayerNorm: 2-5                    256
│    └─LayerNorm: 2-6                    256
│    └─LayerNorm: 2-7                    256
│    └─LayerNorm: 2-8                    256
│    └─LayerNorm: 2-9                    256
│    └─LayerNorm: 2-10                   256
│    └─LayerNorm: 2-11                   256
│    └─LayerNorm: 2-12                   256
│    └─LayerNorm: 2-13                   256
│    └─LayerNorm: 2-14                 